In [3]:
# Cell 1 — download CCLE EPIC 850k methylation data
import requests
from pathlib import Path
from tqdm import tqdm
import pandas as pd

data_dir = Path("/home/bmurnyak/methylation-tissue-classifier/data")
data_dir.mkdir(parents=True, exist_ok=True)

# CCLE methylation data from DepMap portal
files_to_download = [
    {
        'url' : 'https://depmap.org/portal/api/downloads/file?file_name=CCLE_RRBS_TSS_1kb_20181022.txt.gz',
        'name': 'CCLE_methylation.txt.gz',
        'desc': 'CCLE RRBS methylation data'
    },
    {
        'url' : 'https://depmap.org/portal/api/downloads/file?file_name=sample_info.csv',
        'name': 'CCLE_sample_info.csv',
        'desc': 'CCLE sample metadata'
    }
]

for f in files_to_download:
    out = data_dir / f['name']
    if out.exists():
        print(f"  ✓ exists: {f['name']}")
        continue
    print(f"Downloading {f['desc']}...")
    r = requests.get(f['url'], stream=True, timeout=60)
    total = int(r.headers.get('content-length', 0))
    with open(out, 'wb') as fp, tqdm(
        desc=f['name'], total=total, unit='B', unit_scale=True
    ) as bar:
        for chunk in r.iter_content(chunk_size=8192):
            fp.write(chunk)
            bar.update(len(chunk))
    print(f"  ✓ done")

print("\n✓ Downloads complete")

ModuleNotFoundError: No module named 'requests'

In [ ]:
import requests, pandas, numpy, matplotlib, seaborn
print("All OK")
print(f"pandas: {pandas.__version__}")
print(f"numpy: {numpy.__version__}")

In [2]:
# Cell 3 — download TCGA 450k methylation via GDC API (proven approach)
import requests
import json
from pathlib import Path
from tqdm import tqdm

GDC_API = "https://api.gdc.cancer.gov"
data_dir = Path("/home/bmurnyak/methylation-tissue-classifier/data/tcga")
data_dir.mkdir(parents=True, exist_ok=True)

# Cancer types to download — 5 samples each to keep it fast
cancer_types = {
    'BRCA': 'TCGA-BRCA',  # Breast
    'COAD': 'TCGA-COAD',  # Colon
    'LUAD': 'TCGA-LUAD',  # Lung
    'PRAD': 'TCGA-PRAD',  # Prostate
    'LAML': 'TCGA-LAML',  # Blood/AML
}

def get_methylation_files(project, size=5):
    filters = {
        "op": "and",
        "content": [
            {"op": "=", "content": {"field": "cases.project.project_id", "value": project}},
            {"op": "=", "content": {"field": "files.data_type", "value": "Methylation Beta Value"}},
            {"op": "=", "content": {"field": "files.platform", "value": "Illumina Human Methylation 450"}},
            {"op": "=", "content": {"field": "files.access", "value": "open"}}
        ]
    }
    params = {
        "filters": json.dumps(filters),
        "fields" : "file_id,file_name",
        "size"   : size,
        "format" : "JSON"
    }
    r = requests.get(f"{GDC_API}/files", params=params, timeout=30)
    return r.json()['data']['hits']

def download_file(file_id, file_name, output_dir):
    out = Path(output_dir) / file_name
    if out.exists():
        print(f"  ✓ exists: {file_name[:50]}")
        return out
    r = requests.get(f"https://api.gdc.cancer.gov/data/{file_id}", stream=True)
    total = int(r.headers.get('content-length', 0))
    with open(out, 'wb') as f, tqdm(desc=file_name[:40], total=total, unit='B', unit_scale=True) as bar:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))
    return out

# Download 5 samples per cancer type
all_files = {}
for cancer, project in cancer_types.items():
    print(f"\nFetching {cancer} ({project})...")
    cancer_dir = data_dir / cancer
    cancer_dir.mkdir(exist_ok=True)
    files = get_methylation_files(project, size=5)
    print(f"  Found {len(files)} files — downloading...")
    paths = [download_file(f['file_id'], f['file_name'], cancer_dir) for f in files]
    all_files[cancer] = paths
    print(f"  ✓ {cancer}: {len(paths)} files ready")

print("\n✓ All downloads complete")
for cancer, paths in all_files.items():
    print(f"  {cancer}: {len(paths)} samples")

ModuleNotFoundError: No module named 'requests'

In [7]:
# Cell 4 — load all cancer types + GBM and build combined matrix
import pandas as pd
import numpy as np
from pathlib import Path

def load_methylation(filepath, sample_name):
    df = pd.read_csv(filepath, sep='\t', header=None, index_col=0)
    df.columns = [sample_name]
    df.index.name = 'cpg_site'
    return df

data_dir = Path("/home/bmurnyak/methylation-tissue-classifier/data/tcga")

# All cancer types including GBM from glioma project
cancer_dirs = {
    'BRCA' : data_dir / 'BRCA',
    'COAD' : data_dir / 'COAD',
    'LUAD' : data_dir / 'LUAD',
    'PRAD' : data_dir / 'PRAD',
    'LAML' : data_dir / 'LAML',
    'GBM'  : Path("/home/bmurnyak/glioma-meth-pipeline/data/tcga/gbm"),
}

all_dfs = []
sample_meta = []

for cancer, cdir in cancer_dirs.items():
    files = sorted(cdir.glob("*.met")) + sorted(cdir.glob("*.txt"))
    files = files[:5]  # max 5 per type
    print(f"Loading {cancer}: {len(files)} files...")
    for i, f in enumerate(files):
        sample_name = f"{cancer}_{i+1}"
        df = load_methylation(f, sample_name)
        all_dfs.append(df)
        sample_meta.append({'sample': sample_name, 'cancer_type': cancer})

# Combine
meth_matrix = pd.concat(all_dfs, axis=1)
meta_df = pd.DataFrame(sample_meta)

# QC
meth_clean = meth_matrix.dropna()
print(f"\n✓ Combined matrix: {meth_matrix.shape[0]:,} CpGs × {meth_matrix.shape[1]} samples")
print(f"✓ After dropna: {meth_clean.shape[0]:,} CpGs")
print(f"\nSamples per cancer type:")
print(meta_df['cancer_type'].value_counts())

Loading BRCA: 5 files...
Loading COAD: 5 files...
Loading LUAD: 5 files...
Loading PRAD: 5 files...
Loading LAML: 5 files...
Loading GBM: 5 files...

✓ Combined matrix: 486,427 CpGs × 30 samples
✓ After dropna: 370,566 CpGs

Samples per cancer type:
cancer_type
BRCA    5
COAD    5
LUAD    5
PRAD    5
LAML    5
GBM     5
Name: count, dtype: int64


In [8]:
# Cell 5 — define gene panel and extract CpG sites
import pandas as pd
import numpy as np

# Gene panel — 25 genes with tissue-specific methylation
gene_panel = {
    # Tumor suppressors
    'RASSF1A': {'category': 'Tumor suppressor', 'tissue_bias': 'Pan-cancer'},
    'APC'    : {'category': 'Tumor suppressor', 'tissue_bias': 'Colon'},
    'MLH1'   : {'category': 'Tumor suppressor', 'tissue_bias': 'Colon'},
    'BRCA1'  : {'category': 'Tumor suppressor', 'tissue_bias': 'Breast'},
    'SFRP1'  : {'category': 'Tumor suppressor', 'tissue_bias': 'Breast/Colon'},
    'CDH1'   : {'category': 'Tumor suppressor', 'tissue_bias': 'Epithelial'},
    'PTEN'   : {'category': 'Tumor suppressor', 'tissue_bias': 'Pan-cancer'},
    'RB1'    : {'category': 'Tumor suppressor', 'tissue_bias': 'Pan-cancer'},
    'CDKN2A' : {'category': 'Cell cycle',       'tissue_bias': 'Pan-cancer'},
    # Tissue-specific TFs
    'HOXA5'  : {'category': 'Tissue-specific TF', 'tissue_bias': 'Breast'},
    'CDX2'   : {'category': 'Tissue-specific TF', 'tissue_bias': 'Colon'},
    'PAX6'   : {'category': 'Tissue-specific TF', 'tissue_bias': 'Brain'},
    'GATA4'  : {'category': 'Tissue-specific TF', 'tissue_bias': 'GI/Cardiac'},
    'NKX2-1' : {'category': 'Tissue-specific TF', 'tissue_bias': 'Lung'},
    # Cancer biomarkers
    'GSTP1'  : {'category': 'Cancer biomarker', 'tissue_bias': 'Prostate'},
    'MGMT'   : {'category': 'Cancer biomarker', 'tissue_bias': 'Brain'},
    'SEPT9'  : {'category': 'Cancer biomarker', 'tissue_bias': 'Colon'},
    'SHOX2'  : {'category': 'Cancer biomarker', 'tissue_bias': 'Lung'},
    'VIM'    : {'category': 'Cancer biomarker', 'tissue_bias': 'Mesenchymal'},
    # Immune/checkpoint
    'CD274'  : {'category': 'Immune checkpoint', 'tissue_bias': 'Pan-cancer'},
    # DNA repair
    'PARP1'  : {'category': 'DNA repair',       'tissue_bias': 'Brain'},
    # Signaling
    'KRAS'   : {'category': 'Signaling',         'tissue_bias': 'Pan-cancer'},
    # Metabolism
    'PCSK9'  : {'category': 'Metabolism',        'tissue_bias': 'Liver/Pan-cancer'},
    # Blood specific
    'RUNX1'  : {'category': 'Hematopoietic TF', 'tissue_bias': 'Blood'},
    'TET2'   : {'category': 'Epigenetic',        'tissue_bias': 'Blood'},
}

print(f"Gene panel: {len(gene_panel)} genes")
print(f"\nBy category:")
cats = {}
for gene, info in gene_panel.items():
    cats.setdefault(info['category'], []).append(gene)
for cat, genes in cats.items():
    print(f"  {cat}: {', '.join(genes)}")

# Load manifest
manifest = pd.read_csv(
    "/home/bmurnyak/glioma-meth-pipeline/data/manifest_450k.csv",
    skiprows=7, low_memory=False
)[['IlmnID','CHR','MAPINFO','UCSC_RefGene_Name']].rename(columns={
    'IlmnID'            : 'cpg_site',
    'CHR'               : 'chr',
    'MAPINFO'           : 'position',
    'UCSC_RefGene_Name' : 'gene'
})
manifest['gene_clean'] = manifest['gene'].fillna('').apply(
    lambda x: x.split(';')[0] if x else ''
)

# Find CpG sites for each gene
print(f"\nCpG sites per gene on 450k array:")
gene_cpg_map = {}
for gene in gene_panel.keys():
    cpgs = manifest[manifest['gene_clean'] == gene]['cpg_site'].tolist()
    available = [c for c in cpgs if c in meth_clean.index]
    gene_cpg_map[gene] = available
    status = f"{len(available)} sites" if available else "NOT ON ARRAY"
    print(f"  {gene:<12}: {status}")

Gene panel: 25 genes

By category:
  Tumor suppressor: RASSF1A, APC, MLH1, BRCA1, SFRP1, CDH1, PTEN, RB1
  Cell cycle: CDKN2A
  Tissue-specific TF: HOXA5, CDX2, PAX6, GATA4, NKX2-1
  Cancer biomarker: GSTP1, MGMT, SEPT9, SHOX2, VIM
  Immune checkpoint: CD274
  DNA repair: PARP1
  Signaling: KRAS
  Metabolism: PCSK9
  Hematopoietic TF: RUNX1
  Epigenetic: TET2

CpG sites per gene on 450k array:
  RASSF1A     : NOT ON ARRAY
  APC         : 33 sites
  MLH1        : 32 sites
  BRCA1       : 22 sites
  SFRP1       : 33 sites
  CDH1        : 19 sites
  PTEN        : 51 sites
  RB1         : 36 sites
  CDKN2A      : 3 sites
  HOXA5       : 46 sites
  CDX2        : 17 sites
  PAX6        : 94 sites
  GATA4       : 54 sites
  NKX2-1      : 19 sites
  GSTP1       : 10 sites
  MGMT        : 135 sites
  SEPT9       : 149 sites
  SHOX2       : 29 sites
  VIM         : 19 sites
  CD274       : 4 sites
  PARP1       : 12 sites
  KRAS        : 21 sites
  PCSK9       : 21 sites
  RUNX1       : 43 sites

In [14]:
# Cell 6 — build gene-level methylation matrix
import pandas as pd
import numpy as np

# Add RARB to gene_cpg_map
rarb_cpgs = manifest[manifest['gene_clean'] == 'RARB']['cpg_site'].tolist()
gene_cpg_map['RARB'] = [c for c in rarb_cpgs if c in meth_clean.index]
print(f"RARB: {len(gene_cpg_map['RARB'])} CpG sites")

# Remove RASSF1A if present
gene_cpg_map.pop('RASSF1A', None)

# Build mean methylation matrix per gene
print("\nBuilding gene-level methylation matrix...")
gene_meth_rows = []
for gene, cpgs in gene_cpg_map.items():
    if not cpgs:
        continue
    gene_mean = meth_clean.loc[cpgs].mean(axis=0)
    gene_mean.name = gene
    gene_meth_rows.append(gene_mean)

gene_meth = pd.DataFrame(gene_meth_rows)
print(f"✓ Gene methylation matrix: {gene_meth.shape[0]} genes × {gene_meth.shape[1]} samples")
print(f"\nPreview (first 5 genes, 6 samples):")
print(gene_meth.iloc[:5, :6].round(3).to_string())

RARB: 19 CpG sites

Building gene-level methylation matrix...
✓ Gene methylation matrix: 25 genes × 30 samples

Preview (first 5 genes, 6 samples):
       BRCA_1  BRCA_2  BRCA_3  BRCA_4  BRCA_5  COAD_1
APC     0.362   0.272   0.157   0.466   0.190   0.175
MLH1    0.229   0.151   0.164   0.164   0.254   0.161
BRCA1   0.416   0.416   0.418   0.375   0.524   0.435
SFRP1   0.444   0.585   0.216   0.432   0.418   0.535
CDH1    0.428   0.327   0.402   0.351   0.309   0.364


In [15]:
# Cell 7 — tissue-specific methylation heatmap
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Organize columns by cancer type
cancer_order = ['GBM', 'BRCA', 'COAD', 'LUAD', 'PRAD', 'LAML']
col_order = []
for cancer in cancer_order:
    cols = [c for c in gene_meth.columns if c.startswith(cancer)]
    col_order.extend(sorted(cols))

gene_meth_ordered = gene_meth[col_order]

# Row order by category
category_order = [
    'Tumor suppressor', 'Cell cycle', 'Tissue-specific TF',
    'Cancer biomarker', 'Immune checkpoint', 'DNA repair',
    'Signaling', 'Metabolism', 'Hematopoietic TF', 'Epigenetic'
]

# Build gene order
gene_order = []
for cat in category_order:
    genes_in_cat = [g for g, info in gene_panel.items() if info['category'] == cat and g in gene_meth.index]
    gene_order.extend(genes_in_cat)
# Add RARB
if 'RARB' not in gene_order and 'RARB' in gene_meth.index:
    gene_order.append('RARB')

gene_meth_plot = gene_meth_ordered.loc[gene_order]

# Color bars
cancer_colors = {
    'GBM' : '#8B0000', 'BRCA': '#C71585', 'COAD': '#1A5276',
    'LUAD': '#0F6E56', 'PRAD': '#534AB7', 'LAML': '#BA7517'
}
col_colors = [cancer_colors[c.split('_')[0]] for c in col_order]

# Category colors for row annotation
cat_colors_map = {
    'Tumor suppressor'  : '#E24B4A',
    'Cell cycle'        : '#C0392B',
    'Tissue-specific TF': '#185FA5',
    'Cancer biomarker'  : '#0F6E56',
    'Immune checkpoint' : '#534AB7',
    'DNA repair'        : '#BA7517',
    'Signaling'         : '#E67E22',
    'Metabolism'        : '#16A085',
    'Hematopoietic TF'  : '#8E44AD',
    'Epigenetic'        : '#2C3E50'
}

fig, ax = plt.subplots(figsize=(14, 12))

sns.heatmap(
    gene_meth_plot,
    cmap='RdBu_r',
    vmin=0, vmax=1,
    linewidths=0.3,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Mean beta value (methylation)', 'shrink': 0.5},
    yticklabels=True,
    xticklabels=False
)

# Column color bar (cancer type)
for i, color in enumerate(col_colors):
    ax.add_patch(plt.Rectangle(
        (i, len(gene_order) + 0.2), 1, 0.8,
        color=color, clip_on=False,
        transform=ax.get_xaxis_transform()
    ))

# Row category color bar
for i, gene in enumerate(gene_order):
    cat = gene_panel.get(gene, {}).get('category', 'Epigenetic')
    color = cat_colors_map.get(cat, '#888')
    ax.add_patch(plt.Rectangle(
        (-0.4, i), 0.35, 1,
        color=color, clip_on=False,
        transform=ax.get_yaxis_transform()
    ))

# Cancer type labels below color bar
x_pos = 0
for cancer in cancer_order:
    n = sum(1 for c in col_order if c.startswith(cancer))
    ax.text(x_pos + n/2, len(gene_order) + 1.2, cancer,
            ha='center', va='bottom', fontsize=9,
            fontweight='bold', color=cancer_colors[cancer],
            transform=ax.get_xaxis_transform())
    x_pos += n

ax.set_title('Tissue-Specific Methylation Panel\n25 genes × 6 cancer types (TCGA, n=5 per type)',
             fontsize=13, fontweight='bold', pad=20)
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=9)

# Category legend
from matplotlib.patches import Patch
legend_patches = [Patch(color=c, label=cat) for cat, c in cat_colors_map.items()
                  if any(gene_panel.get(g, {}).get('category') == cat for g in gene_order)]
ax.legend(handles=legend_patches, loc='upper left',
          bbox_to_anchor=(1.15, 1), fontsize=8, title='Category',
          title_fontsize=9)

plt.tight_layout()
plt.savefig('/home/bmurnyak/methylation-tissue-classifier/results/plots/tissue_methylation_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✓ Heatmap saved")

ModuleNotFoundError: No module named 'matplotlib'